In [ ]:
import os
import torch
import numpy as np
import nibabel as nib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "monai", "nibabel"], check=True)

import monai
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    ScaleIntensityRangePercentilesd, RandCropByPosNegLabeld,
    RandFlipd, RandRotate90d, RandGaussianSmoothd, ToTensord
)
from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference

print("MONAI version:", monai.__version__)

In [ ]:
base_dir = r"" #path to the data
patient_data = []

patient_nums = []
for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue
    patient_num = folder_name.split("-")[1]
    patient_nums.append(int(patient_num))

patient_nums.sort()
for p_num in patient_nums:
    flair_path = os.path.join(base_dir, f"Patient-{p_num}", f"{p_num}-Flair.nii")
    label_path = os.path.join(base_dir, f"Patient-{p_num}", f"{p_num}-LesionSeg-Flair.nii")
    if os.path.exists(flair_path) and os.path.exists(label_path):
        patient_data.append({"image": flair_path, "label": label_path})

total_patients = len(patient_data)
split_idx = int(0.8 * total_patients)
train_files = patient_data[:split_idx]
val_files = patient_data[split_idx:]

print(f"Total Patients: {total_patients}, Train: {len(train_files)}, Val: {len(val_files)}")

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(128, 128, 16),
        pos=1, neg=1, num_samples=4,
        image_key="image", image_threshold=0
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    ToTensord(keys=["image", "label"])
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
    ToTensord(keys=["image", "label"])
])

train_ds = Dataset(data=train_files, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)

val_ds = Dataset(data=val_files, transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=2, pin_memory=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_function = DiceCELoss(to_onehot_y=False, sigmoid=True, squared_pred=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
dice_metric = DiceMetric(include_background=True, reduction="mean")
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
max_epochs = 150
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

post_pred = monai.transforms.Compose([monai.transforms.Activations(sigmoid=True), monai.transforms.AsDiscrete(threshold=0.5)])

for epoch in range(1, max_epochs + 1):
    model.train()
    epoch_loss = 0
    step = 0
    with tqdm(train_loader, desc=f"Epoch {epoch}/{max_epochs} Train") as pbar:
        for batch_data in pbar:
            step += 1
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            
            
            labels = (labels > 0).float()
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
            
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    
    if epoch % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                val_labels = (val_labels > 0).float()
                
                
                roi_size = (128, 128, 16)
                sw_batch_size = 4
                val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)
                
                val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
                val_labels = decollate_batch(val_labels)
                
                dice_metric(y_pred=val_outputs, y=val_labels)
                
            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)
            
            scheduler.step()
            print(f"Epoch {epoch} Val Dice: {metric:.4f} | Train Loss: {epoch_loss:.4f}")
            
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch
                torch.save(model.state_dict(), "best_metric_3d_unet.pth")
                print("  => saved new best metric model")
                
print(f"Train completed. Best Dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.title("Train DiceCELoss")
plt.plot(range(1, max_epochs + 1), epoch_loss_values, label="Train Loss")
plt.subplot(1, 2, 2)
plt.title("Val Mean Dice")
val_epochs = [i * val_interval for i in range(1, len(metric_values) + 1)]
plt.plot(val_epochs, metric_values, color="orange", marker="o", label="Val Dice")
plt.show()